# Deep-Dive Conceptual Roadmap & Dataset Ecosystem

## Technical Terminology & Mechanics

While LoRA modifies the internal weight matrices of a model, **Soft Prompting** techniques (specifically **Prompt Tuning** and **Prefix Tuning**) leave the entire pre-trained model architecture completely frozen. Instead, they optimize a small set of continuous, trainable tensors appended to the data stream.

### Prompt Tuning

Prepends a sequence of trainable continuous embeddings (**Virtual Tokens**, $P_E$) directly to the input sequence $X$ at the very first layer. The model processes:

$$X' = [P_E; X]$$

### Prefix Tuning

Prepends trainable continuous prefix vectors to the **Key** ($K$) and **Value** ($V$) matrices within the attention mechanism of **every single transformer block**. The modified attention equation becomes:

$$\text{head} = \text{Attention}(Q, [P_K; K], [P_V; V])$$

---

## Crisp Definition

Prompt and Prefix Tuning solve the brittleness of manual **"hard prompting"** (guessing the right words) by allowing gradient descent to mathematically compute the optimal, continuous instruction vectors in the model's **native embedding space**.

---

## The Human Element: Industry-Standard Dataset Ecosystem

Because soft prompting essentially acts as an invisible, mathematically perfect **"System Prompt"**, the datasets used for this technique focus heavily on **persona adoption**, **style transfer**, and **strict behavioral guidelines** rather than raw knowledge ingestion.

1. `fka/awesome-chatgpt-prompts`

    A curated list of explicit role-playing instructions (e.g., `"Act as a Linux Terminal"`, `"Act as a JavaScript Console"`).

    **PEFT Alignment:** Prompt Tuning is designed to compress highly specific behavioral instructions into virtual tokens. This dataset trains the virtual tokens to **permanently lock the model into a specific persona** without altering its foundational knowledge.

2. `HuggingFaceH4/ultrafeedback_binarized`

    A dataset containing pairs of **chosen and rejected** assistant responses, emphasizing tone, safety, and helpfulness.

    **PEFT Alignment:** Prefix Tuning is highly effective at steering response generation quality because it injects constraints **at every layer**. The high variance in response quality allows the prefix gradients to map exactly how to guide the internal attention layers toward safer, higher-quality tokens.

---

# Architectural Context Block

## The "Why"

Human language is **discrete**, but neural networks operate in **continuous vector spaces**. When you write a prompt like `"Translate to French"`, you are restricted by the model's existing tokenizer dictionary.

Prompt/Prefix Tuning removes this linguistic barrier. By optimizing in the continuous embedding space, backpropagation can discover **"words" (virtual tokens) that do not exist in any human language** but perfectly trigger the exact neural pathways required to solve the task.

---

## VRAM & Compute Impact

### Parameter Extreme Efficiency

Prompt Tuning is the **most parameter-efficient technique in existence**. If you use **20 virtual tokens** on a model with a hidden size of **1536**, you are only training:

$$20 \times 1536 = 30{,}720 \text{ parameters}$$

This is **orders of magnitude smaller** than even LoRA (~15 million parameters for the same model).

### Storage Footprint

A fully trained Prompt Tuning adapter checkpoint is often **less than 500 Kilobytes**, allowing you to store thousands of specialized task adapters on a single thumb drive.

### Context Window Cost

The virtual tokens consume space in your maximum context window. For example:

- **Max sequence length:** 1024 tokens
- **Virtual tokens used:** 20
- **Effective context window:** 1004 tokens

---

## Architectural Trade-offs

### ✅ Pros

- **Zero Base Degradation:** The base weights are **100% frozen**. There is zero risk of catastrophic forgetting regarding the model's core knowledge.

- **Batch Task Routing:** In production, you can process a single batch of mixed requests where:
  - **Request A** uses a `"Translation"` soft prompt
  - **Request B** uses a `"Summarization"` soft prompt

  Both are pushed through the **same frozen model simultaneously**.

### ❌ Cons

- **Slower Convergence:** Because you are only updating a microscopic fraction of parameters at the very beginning of the network, the gradients have a long way to travel. Prompt Tuning requires:
  - Significantly **higher learning rates** (e.g., `1e-3` to `3e-2`)
  - **More epochs** to converge compared to LoRA

# Production-Grade Code / Configuration

The following script sets up **Prompt Tuning** on your **Google Colab T4**. Notice the dramatic shift in the **learning rate** and the **initialization text** used to "kickstart" the virtual tokens.

## Environment Setup

In [ ]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%pip install torchao==0.16.0 transformers trl peft accelerate bitsandbytes datasets

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from peft import PromptTuningConfig, PromptTuningInit, TaskType
from trl import SFTConfig, SFTTrainer

# ---
# 1. Environment & Target Entities
# ---
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "HuggingFaceH4/ultrafeedback_binarized"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

## Data Preparation

In [ ]:
raw_dataset = load_dataset(DATASET_ID, split="train_prefs[:200]")

def format_ultrafeedback(batch):
    prompts, completions = [], []

    # UltraFeedback uses parallel arrays for 'prompt', 'chosen', and 'rejected'
    for i in range(len(batch['prompt'])):
        user_prompt = batch['prompt'][i]

        # 'chosen' is a list of dictionaries: [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
        # We grab the last item [-1] to get the final, highest-scored assistant response
        best_response = batch['chosen'][i][-1]['content']

        # We inject the overarching persona we want the virtual tokens to learn
        prompts.append(f"### Input:\n[System: Act as an elite, highly detailed AI Assistant]\nInstruction: {user_prompt}\n\n")

        # The target completion is now the GPT-4 curated response
        completions.append(f"### Response:\n{best_response}")

    return {"prompt": prompts, "completion": completions}

processed_dataset = raw_dataset.map(format_ultrafeedback, batched=True)

## Model Training

In [ ]:
# ---
# 3. prompt tuning config
# ---
INIT_TEXT = "Act as an obedient, highly capable AI assistant. Strictly follow instructions."

peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    prompt_tuning_init_text=INIT_TEXT,
    num_virtual_tokens=len(tokenizer(INIT_TEXT)["input_ids"]), # Maps exactly to the init text length
    tokenizer_name_or_path=MODEL_ID,
)

# ---
# 3. base model config
# ---
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
    trust_remote_code=True
)

In [ ]:
# ---
# 4. Production Training Hyperparameters
# ---
training_args = SFTConfig(
    output_dir="./qwen_prompt_t4",
    run_name="qwen_prompt_t4",

    # Batch & Gradient
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,

    # Sequence & Packing
    max_length=512,
    truncation_mode="keep_start",
    packing=False,
    completion_only_loss=True,

    # Precision
    fp16=True,
    bf16=False,

    # Optimizer & Learning Rate
    optim="paged_adamw_8bit",
    learning_rate=1e-2,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.001,
    max_grad_norm=0.3,

    # Memory Optimization
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    torch_empty_cache_steps=25,

    # Training Duration
    max_steps=150,
    num_train_epochs=2,

    # Logging
    logging_strategy="steps",
    logging_steps=5,
    logging_first_step=True,
    report_to="none",

    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=1,

    # Dataset
    dataset_num_proc=2,
    dataset_kwargs={
        "add_special_tokens": False,
        "skip_prepare_dataset": False,
    },

    # Reproducibility
    seed=42,
    data_seed=42,
    shuffle_dataset=True,

    # Performance
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
)

# ---
# 6. Trainer Execution
# ---
trainer = SFTTrainer(
    model=base_model,
    train_dataset=processed_dataset,
    peft_config=peft_config,
    args=training_args,
    processing_class=tokenizer,
)


In [ ]:
trainer.train()

print("[Success] fine-tuning complete. Saving adapter weights...")
trainer.model.save_pretrained("./peft_prompt_adapter")

### To download fine-tuned model

In [ ]:
import shutil
from google.colab import files

# Name of the folder you want to download
folder_to_zip = './peft_prompt_adapter'
# Name of the resulting zip file
output_filename = 'peft_prompt_adapter.zip'

# Create the zip archive
shutil.make_archive('peft_prompt_adapter', 'zip', folder_to_zip)

# Download the file to your machine
# files.download(output_filename)

In [ ]:
# ---
# To Save Model to Google Drive
# ---
from google.colab import drive
import shutil
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the destination path in your Drive
destination_folder = '/content/drive/MyDrive/colab_models'
os.makedirs(destination_folder, exist_ok=True)

source_path = '/content/peft_prompt_adapter.zip'
destination_path = os.path.join(destination_folder, 'peft_prompt_adapter.zip')

# 3. Copy the file
print(f"Copying {source_path} to {destination_path}...")
shutil.copy(source_path, destination_path)
print("Done! You can now find the model in your Google Drive under 'colab_models'.")

# Model Usage

In [ ]:
import torch
import time
import warnings
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# ==========================================
# 1. Environment & Warning Management
# ==========================================
# Suppress the harmless PEFT positional IDs warning to maintain clean logs
warnings.filterwarnings("ignore", message=".*Position ids are not supported for parameter efficient tuning.*")

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "./peft_prompt_adapter"  # Path where your new adapter is saved

print("[Init] Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ==========================================
# 2. Ingest Base Model Weights
# ==========================================
print("[Init] Loading Base Model into VRAM (FP16)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
    trust_remote_code=True
)

# ==========================================
# 3. Dedicated Evaluation Pipeline
# ==========================================
def evaluate_model(inference_model, query_text):
    """
    Wraps the input query in the exact structural context used during
    the UltraFeedback training phase to trigger the soft prompt embeddings.
    """
    formatted_prompt = (
        f"### Input:\n"
        f"[System: Act as an elite, highly detailed AI Assistant]\n"
        f"Instruction: {query_text}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(inference_model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=300,        # Higher token budget to allow the rich UltraFeedback style
            temperature=0.4,           # Balanced creativity and structural consistency
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - start_time

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Isolate the text generated strictly after the prompt boundary
    clean_response = full_text.split("### Response:\n")[-1].strip()
    return clean_response, latency

In [ ]:
TEST_PROMPT = "Explain the core architectural differences between standard LoRA and Prompt Tuning."

print("\n" + "="*70)
print(f"EVALUATION PROMPT: {TEST_PROMPT}")
print("="*70 + "\n")

# 4a. Execute Baseline
print(">>> BASELINE UNTUNED MODEL RESPONSE <<<")
base_output, base_time = evaluate_model(model, TEST_PROMPT)
print(base_output)
print(f"\n[Base Model Latency: {base_time:.2f}s]\n")
print("-" * 70 + "\n")

# 4b. Load Virtual Tokens and Execute Tuned Model
print("[Init] Dynamically wrapping base model with UltraFeedback virtual tokens...")
tuned_model = PeftModel.from_pretrained(model, ADAPTER_DIR)

print("\n>>> ULTRAFEEDBACK PROMPT-TUNED RESPONSE <<<")
tuned_output, tuned_time = evaluate_model(tuned_model, TEST_PROMPT)
print(tuned_output)
print(f"\n[Tuned Model Latency: {tuned_time:.2f}s]\n")